# exp094_projection_only_on_exp073 train

Projection-only postprocess audit on the deterministic exp073 OOF prediction.


## Contents

1. Setup and configuration
2. Input and projection contract
3. Run projection grid
4. Preview outputs
5. Metrics and branch decision


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from projection_only_on_exp073 import run_train_from_config, to_jsonable

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()
config.setdefault("runtime", {})["output_dir"] = str(paths.artifacts_dir)
config.setdefault("data", {})["raw_dir"] = str(paths.raw_data_dir)
config.setdefault("data", {})["train_dir"] = str(paths.train_data_dir)
config.setdefault("data", {})["test_dir"] = str(paths.test_data_dir)
config.setdefault("data", {})["sample_submission"] = str(paths.sample_submission_path)

if DEBUG:
    config.setdefault("audit", {})["max_rows"] = int(os.environ.get("EXPERIMENT_MAX_ROWS", "50000"))

print(json.dumps({
    "experiment": EXPERIMENT_NAME,
    "route": get_nested(config, "experiment.route"),
    "status": get_nested(config, "experiment.status"),
    "parent": get_nested(config, "lineage.parent"),
    "mode": get_nested(config, "audit.mode"),
    "selected_mode": get_nested(config, "audit.selected_mode"),
    "selected_model": get_nested(config, "audit.selected_model"),
    "debug": DEBUG,
    "max_rows": get_nested(config, "audit.max_rows"),
    "train_dir": get_nested(config, "data.train_dir"),
    "artifacts_dir": str(paths.artifacts_dir),
}, indent=2, sort_keys=True))


## 2. Input and projection contract


In [ ]:
print(json.dumps({
    "exp073_oof_predictions": get_nested(config, "data.exp073_oof_predictions"),
    "train_dir": get_nested(config, "data.train_dir"),
    "grid": get_nested(config, "audit.grid"),
    "selection_guard": get_nested(config, "audit.selection_guard"),
    "leakage_policy": get_nested(config, "validation.leakage_policy"),
    "expected_train_artifacts": get_nested(config, "audit.expected_train_artifacts"),
}, indent=2, ensure_ascii=False))

for label, values in {
    "exp073_oof_predictions": get_nested(config, "data.exp073_oof_predictions"),
}.items():
    for value in values:
        path = Path(value)
        print(label, value, "exists=" + str(path.exists()), "size=" + str(path.stat().st_size if path.exists() else 0))


## 3. Run projection grid


In [ ]:
summary = run_train_from_config(config)
print(json.dumps(to_jsonable({
    "status": summary["status"],
    "runtime_seconds": summary["runtime_seconds"],
    "source_rows": summary["source"]["exp073_oof_predictions"]["rows"],
    "source_wells": summary["source"]["exp073_oof_predictions"]["wells"],
    "baseline_rmse": summary["baseline"]["rmse_tvt"],
    "best_variant": summary["best_variant"],
    "guard": summary["guard"],
}), indent=2, sort_keys=True))


## 4. Preview outputs


In [ ]:
artifact_paths = {name: paths.artifacts_dir / filename for name, filename in summary["outputs"].items() if filename}
for name, path in artifact_paths.items():
    print(f"{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

def preview_csv(name: str, n: int = 10) -> pd.DataFrame:
    path = artifact_paths[name]
    frame = pd.read_csv(path, nrows=n)
    display(frame)
    return frame

variant_preview = preview_csv("variant_metrics")
fold_preview = preview_csv("fold_metrics")
bucket_preview = preview_csv("bucket_metrics")
by_well_preview = preview_csv("by_well")


## 5. Metrics and branch decision


In [ ]:
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": summary["status"],
    "updated_at": datetime.now(UTC).isoformat(),
    "route": get_nested(config, "experiment.route"),
    "metric": get_nested(config, "validation.metric"),
    "parent": get_nested(config, "lineage.parent"),
    "source": summary["source"],
    "baseline": summary["baseline"],
    "best_variant": summary["best_variant"],
    "guard": summary["guard"],
    "grid": summary["grid"],
    "outputs": summary["outputs"],
}
metrics_path = paths.experiment_dir / "metrics.json"
with metrics_path.open("w") as fp:
    json.dump(to_jsonable(metrics), fp, indent=2, sort_keys=True)
print(json.dumps(to_jsonable(metrics), indent=2, sort_keys=True))
